# Interface check on CAGE-2
A quick manual check of the reduced interface: a scripted red kill chain (user footholds, enterprise discovery, exploitation, escalation, and Impact), then every blue action in turn, then the enterprise chain again. Each step prints the action, whether CybORG reports success, the true state, and the tracked state vectors.

In [ ]:
%run Initialize.ipynb

## Scripted red kill chain from a fresh reset

In [ ]:
Steps = [3, 4, 5, 6, 9, 10, 11, 12, 1, 2, 7, 8, 13, 14, 15, 16]
blue_state, blue_knowledge, red_state, red_knowledge, real_state = Reset_Episode()
for i, idx in enumerate(Steps):
    action_name, cyborg_action = Red_Action_Gen(idx, red_knowledge, [0, 0])
    res_red = cyborg.step(agent='Red', action=cyborg_action)
    real_state = System_Real_State(cyborg.environment_controller.state)
    red_knowledge = red_knowledge_dic(res_red.observation, red_knowledge)
    red_state = red_state_vector(res_red.observation, red_knowledge, red_state, action_name)
    print(f"step {i + 1:2d}  {action_name:<34s} success: {res_red.observation['success']}  "
          f"true state: {real_state}  red state: {red_state}")

## Every blue action in turn

In [ ]:
def red_presence():
    state = cyborg.environment_controller.state
    hosts = [h for h in ['User1', 'User2', 'User3', 'User4', 'Enterprise0', 'Enterprise1']
             if len(state.hosts[h].sessions.get('Red', [])) > 0]
    print("red sessions on:", hosts if hosts else "none")

red_presence()
for i in range(n_a):
    action_name, y, real_state = blue_step(i)
    blue_state = blue_state_vector(y, blue_state, action_name, i + 1)
    print(f"step {i + 1:2d}  {action_name:<22s} y: {y}  blue state: {blue_state}  true state: {real_state}")
red_presence()

## The enterprise chain again

In [ ]:
print('initial red state:', red_state)
for i, idx in enumerate([15, 16, 7, 8, 13, 14, 15, 16]):
    action_name, cyborg_action = Red_Action_Gen(idx, red_knowledge, [0, 0])
    res_red = cyborg.step(agent='Red', action=cyborg_action)
    real_state = System_Real_State(cyborg.environment_controller.state)
    red_knowledge = red_knowledge_dic(res_red.observation, red_knowledge)
    red_state = red_state_vector(res_red.observation, red_knowledge, red_state, action_name)
    print(f"step {i + 1:2d}  {action_name:<34s} success: {res_red.observation['success']}  "
          f"true state: {real_state}  red state: {red_state}")